In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# Loading of the data
df = pd.read_csv("players.csv")


# Feature Engineering

df["avg_runs"] = df["total_runs"] / df["outs"]
df["strike_rate"] = (df["total_runs"] / df["balls_faced"]) * 100
df["boundary_runs"] = df["fours"] * 4 + df["sixes"] * 6
df["boundary_pct"] = df["boundary_runs"] / df["total_runs"]

# Fake consistency (since we don't have match-wise data)
# Lower value = more consistent
df["consistency"] = 1 / df["avg_runs"]

# Select features
features = ["avg_runs", "strike_rate", "consistency", "boundary_pct"]
X = df[features]


# Normalize

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Similarity

similarity_matrix = cosine_similarity(X_scaled)

# Function to get similar players
def get_similar_players(player_name, top_n=5):
    if player_name not in df["player_name"].values:
        return "Player not found"

    idx = df[df["player_name"] == player_name].index[0]
    scores = list(enumerate(similarity_matrix[idx]))

    # Sort by similarity
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # Skip itself
    top_players = scores[1:top_n+1]

    results = []
    for i, score in top_players:
        results.append((df.iloc[i]["player_name"], round(score, 3)))

    return results


# Test
if __name__ == "__main__":
    player = input("Enter player name: ")
    similar = get_similar_players(player)

    print("\nTop similar players:")
    for p, score in similar:
        print(f"{p} (Similarity: {score})")


Top similar players:
Glenn Maxwell (Similarity: 0.867)
Andre Russell (Similarity: 0.398)
Rohit Sharma (Similarity: -0.06)
KL Rahul (Similarity: -0.31)
Virat Kohli (Similarity: -0.588)
